# 0. 코랩 준비

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install dpkt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.0/195.0 kB 17.5 MB/s eta 0:00:00


# 1. 12개의 pcap파일 목록

benign파일은 없다.

In [3]:
from pathlib import Path
import pandas as pd

PCAP_DIR = Path("/content/drive/MyDrive/CICIoT2023")

pcap_files = sorted(list(PCAP_DIR.glob("*.pcap")))

print("PCAP 파일 수:", len(pcap_files))

for i, file in enumerate(pcap_files):
    print(i, file.name)

PCAP 파일 수: 12
0 DDoS-ACK_Fragmentation.pcap
1 DDoS-HTTP_Flood-.pcap
2 DDoS-ICMP_Flood.pcap
3 DDoS-ICMP_Fragmentation.pcap
4 DDoS-PSHACK_Flood.pcap
5 DDoS-RSTFINFlood.pcap
6 DDoS-SYN_Flood.pcap
7 DDoS-SlowLoris.pcap
8 DDoS-SynonymousIP_Flood.pcap
9 DDoS-TCP_Flood.pcap
10 DDoS-UDP_Flood.pcap
11 DDoS-UDP_Fragmentation.pcap


# 2. 파일기본정보

단순 파일용량 정리.,필요없

In [4]:
import os

file_info = []

for file in pcap_files:

    file_info.append({
        "file": file.name,
        "size_MB": os.path.getsize(file) / (1024**2)
    })

file_info_df = pd.DataFrame(file_info)

file_info_df

,file,size_MB
0,DDoS-ACK_Fragmentation.pcap,1953.126059
1,DDoS-HTTP_Flood-.pcap,582.558085
2,DDoS-ICMP_Flood.pcap,1953.125066
3,DDoS-ICMP_Fragmentation.pcap,1953.126064
4,DDoS-PSHACK_Flood.pcap,1953.125021
5,DDoS-RSTFINFlood.pcap,1953.125043
6,DDoS-SYN_Flood.pcap,1953.125060
7,DDoS-SlowLoris.pcap,644.309744
8,DDoS-SynonymousIP_Flood.pcap,1953.125023
9,DDoS-TCP_Flood.pcap,1953.125033


# 3. pcap에서 mac/ip/패킷 추출 함수

실질 네트워크 eda


In [5]:
import dpkt
import socket
from collections import defaultdict, Counter


def format_mac(mac_bytes):
    return ":".join(f"{b:02X}" for b in mac_bytes)


def format_ip(ip_bytes):

    try:
        return socket.inet_ntop(socket.AF_INET, ip_bytes)

    except:
        try:
            return socket.inet_ntop(socket.AF_INET6, ip_bytes)
        except:
            return None

In [6]:
#pcap하나 읽어서 기기별 통신량 계산하는 함수

def analyze_pcap_devices(file_path):

    stats = defaultdict(lambda: {
        "sent_packets": 0,
        "received_packets": 0,
        "sent_bytes": 0,
        "received_bytes": 0,
        "ips": set(),
        "peers": Counter()
    })

    f = open(file_path, "rb")

    # pcap / pcapng 둘 다 대응
    try:
        reader = dpkt.pcap.Reader(f)
    except:
        f.seek(0)
        reader = dpkt.pcapng.Reader(f)

    total_packets = 0

    for timestamp, buf in reader:

        try:
            eth = dpkt.ethernet.Ethernet(buf)

            src_mac = format_mac(eth.src)
            dst_mac = format_mac(eth.dst)

            packet_size = len(buf)

            total_packets += 1

            # 송신
            stats[src_mac]["sent_packets"] += 1
            stats[src_mac]["sent_bytes"] += packet_size

            # 수신
            stats[dst_mac]["received_packets"] += 1
            stats[dst_mac]["received_bytes"] += packet_size

            # 상대 기기
            stats[src_mac]["peers"][dst_mac] += 1
            stats[dst_mac]["peers"][src_mac] += 1

            # IPv4
            if isinstance(eth.data, dpkt.ip.IP):

                ip = eth.data

                src_ip = format_ip(ip.src)
                dst_ip = format_ip(ip.dst)

                stats[src_mac]["ips"].add(src_ip)
                stats[dst_mac]["ips"].add(dst_ip)

            # IPv6
            elif isinstance(eth.data, dpkt.ip6.IP6):

                ip = eth.data

                src_ip = format_ip(ip.src)
                dst_ip = format_ip(ip.dst)

                stats[src_mac]["ips"].add(src_ip)
                stats[dst_mac]["ips"].add(dst_ip)

        except:
            continue

    f.close()

    rows = []

    for mac, s in stats.items():

        total = (
            s["sent_packets"]
            + s["received_packets"]
        )

        total_bytes = (
            s["sent_bytes"]
            + s["received_bytes"]
        )

        if len(s["peers"]) > 0:
            top_peer, top_peer_packets = s["peers"].most_common(1)[0]
        else:
            top_peer = None
            top_peer_packets = 0

        rows.append({
            "mac": mac,
            "ips": ", ".join(sorted(x for x in s["ips"] if x)),
            "sent_packets": s["sent_packets"],
            "received_packets": s["received_packets"],
            "total_packets": total,
            "sent_bytes": s["sent_bytes"],
            "received_bytes": s["received_bytes"],
            "total_bytes": total_bytes,
            "unique_peers": len(s["peers"]),
            "top_peer": top_peer,
            "top_peer_packets": top_peer_packets
        })

    df = pd.DataFrame(rows)

    df = df.sort_values(
        "total_packets",
        ascending=False
    )

    return total_packets, df

# 4. pcap하나 먼저 확인

In [7]:
file = pcap_files[0]

total_packets, device_df = analyze_pcap_devices(file)

print("파일:", file.name)
print("전체 패킷:", total_packets)

device_df.head(20)

파일: DDoS-ACK_Fragmentation.pcap
전체 패킷: 2503835


,mac,ips,sent_packets,received_packets,total_packets,sent_bytes,received_bytes,total_bytes,unique_peers,top_peer,top_peer_packets
77,B0:C5:54:59:2E:99,192.168.137.132,8278,878305,886583,516304,826686739,827203043,9,DC:A6:32:C9:E4:AB,221817
68,8C:85:80:6C:B6:47,"192.168.137.123, fe80::8e85:80ff:fe6c:b647",268504,544412,812916,16137169,512315019,528452188,11,DC:A6:32:DC:27:D5,339968
63,DC:A6:32:C9:E4:AB,"0.0.0.0, 192.168.137.196, fd5a:7bd:2aea:e111:d...",555586,56391,611977,522916337,3384066,526300403,9,B0:C5:54:59:2E:99,221817
59,DC:A6:32:DC:27:D5,"0.0.0.0, 192.168.137.65, fd5a:7bd:2aea:e111:de...",486229,114608,600837,457526433,6877616,464404049,9,8C:85:80:6C:B6:47,339968
58,9C:8E:CD:1D:AB:9F,192.168.137.120,12678,564673,577351,1181940,530046438,531228378,9,DC:A6:32:C9:E4:AB,206525
64,DC:A6:32:C9:E4:90,"0.0.0.0, 192.168.137.17, fe80::dea6:32ff:fec9:...",360659,31348,392007,339201719,1882016,341083735,8,B0:C5:54:59:2E:99,187388
62,DC:A6:32:C9:E5:EF,"0.0.0.0, 192.168.137.42, fd5a:7bd:2aea:e111:de...",268720,27981,296701,252612449,1679958,254292407,9,B0:C5:54:59:2E:99,136301
65,E4:5F:01:55:90:C4,"0.0.0.0, 192.168.137.12, fd5a:7bd:2aea:e111:e6...",196733,26868,223601,184786226,1772178,186558404,9,B0:C5:54:59:2E:99,87632
60,DC:A6:32:C9:E4:D5,"0.0.0.0, 192.168.137.20, fd5a:7bd:2aea:e111:de...",129808,14235,144043,121638615,854668,122493283,9,9C:8E:CD:1D:AB:9F,49013
61,DC:A6:32:C9:E5:A4,"0.0.0.0, 192.168.137.186, fe80::dea6:32ff:fec9...",86639,12595,99234,80913576,755738,81669314,8,B0:C5:54:59:2E:99,40354


In [8]:
#FF:FF:FF:FF:FF:FF같은 broadcast mac는 기기가 아니니 제외하기
device_df = device_df[
    device_df["mac"] != "FF:FF:FF:FF:FF:FF"
]

device_df.head(20)

,mac,ips,sent_packets,received_packets,total_packets,sent_bytes,received_bytes,total_bytes,unique_peers,top_peer,top_peer_packets
77,B0:C5:54:59:2E:99,192.168.137.132,8278,878305,886583,516304,826686739,827203043,9,DC:A6:32:C9:E4:AB,221817
68,8C:85:80:6C:B6:47,"192.168.137.123, fe80::8e85:80ff:fe6c:b647",268504,544412,812916,16137169,512315019,528452188,11,DC:A6:32:DC:27:D5,339968
63,DC:A6:32:C9:E4:AB,"0.0.0.0, 192.168.137.196, fd5a:7bd:2aea:e111:d...",555586,56391,611977,522916337,3384066,526300403,9,B0:C5:54:59:2E:99,221817
59,DC:A6:32:DC:27:D5,"0.0.0.0, 192.168.137.65, fd5a:7bd:2aea:e111:de...",486229,114608,600837,457526433,6877616,464404049,9,8C:85:80:6C:B6:47,339968
58,9C:8E:CD:1D:AB:9F,192.168.137.120,12678,564673,577351,1181940,530046438,531228378,9,DC:A6:32:C9:E4:AB,206525
64,DC:A6:32:C9:E4:90,"0.0.0.0, 192.168.137.17, fe80::dea6:32ff:fec9:...",360659,31348,392007,339201719,1882016,341083735,8,B0:C5:54:59:2E:99,187388
62,DC:A6:32:C9:E5:EF,"0.0.0.0, 192.168.137.42, fd5a:7bd:2aea:e111:de...",268720,27981,296701,252612449,1679958,254292407,9,B0:C5:54:59:2E:99,136301
65,E4:5F:01:55:90:C4,"0.0.0.0, 192.168.137.12, fd5a:7bd:2aea:e111:e6...",196733,26868,223601,184786226,1772178,186558404,9,B0:C5:54:59:2E:99,87632
60,DC:A6:32:C9:E4:D5,"0.0.0.0, 192.168.137.20, fd5a:7bd:2aea:e111:de...",129808,14235,144043,121638615,854668,122493283,9,9C:8E:CD:1D:AB:9F,49013
61,DC:A6:32:C9:E5:A4,"0.0.0.0, 192.168.137.186, fe80::dea6:32ff:fec9...",86639,12595,99234,80913576,755738,81669314,8,B0:C5:54:59:2E:99,40354


# 5. 카메라 MAC을 실제 이름으로 매핑

In [9]:
CAMERA_MAP = {

    "9C:8E:CD:1D:AB:9F":
        "AMCREST WiFi Camera",

    "3C:37:86:6F:B9:51":
        "Arlo Base Station",

    "40:5D:82:35:14:C8":
        "Arlo Q Indoor Camera",

    "C0:E7:BF:0A:79:D1":
        "Borun/Sichuan-AI Camera",

    "B0:C5:54:59:2E:99":
        "D-Link Mini Camera",

    "44:01:BB:EC:10:4A":
        "HeimVision Smart WiFi Camera",

    "34:75:63:73:F3:36":
        "Home Eye Camera",

    "7C:A7:B0:CD:18:32":
        "Luohe Cam Dog",

    "44:BB:3B:00:39:07":
        "Nest Indoor Camera",

    "70:EE:50:68:0E:32":
        "Netatmo Camera",

    "10:5A:17:97:A5:C6":
        "Rbcior Camera",

    "10:2C:6B:1B:43:BE":
        "SIMCAM 1S",

    "6C:5A:B0:44:1D:90":
        "TP-Link Tapo Camera",

    "7C:78:B2:86:0D:81":
        "Wyze Camera",

    "84:7A:B6:64:62:58":
        "Yi Indoor Camera",

    "84:7A:B6:62:3A:6C":
        "Yi Indoor 2 Camera",

    "2C:D2:6B:66:D2:87":
        "Yi Outdoor Camera"
}

#해당 pcap에서 어떤 카메라가 얼마나 많이 통신했는지 확인
device_df["camera_name"] = device_df["mac"].map(CAMERA_MAP)

camera_df = device_df[
    device_df["camera_name"].notna()
]

camera_df[
    [
        "camera_name",
        "mac",
        "ips",
        "sent_packets",
        "received_packets",
        "total_packets",
        "total_bytes",
        "top_peer"
    ]
]

,camera_name,mac,ips,sent_packets,received_packets,total_packets,total_bytes,top_peer
77,D-Link Mini Camera,B0:C5:54:59:2E:99,192.168.137.132,8278,878305,886583,827203043,DC:A6:32:C9:E4:AB
58,AMCREST WiFi Camera,9C:8E:CD:1D:AB:9F,192.168.137.120,12678,564673,577351,531228378,DC:A6:32:C9:E4:AB
48,Arlo Q Indoor Camera,40:5D:82:35:14:C8,192.168.137.51,7020,6290,13310,8974607,3C:18:A0:41:C3:A0
2,Nest Indoor Camera,44:BB:3B:00:39:07,"0.0.0.0, 192.168.137.149",3177,2834,6011,1915891,3C:18:A0:41:C3:A0
47,Netatmo Camera,70:EE:50:68:0E:32,"0.0.0.0, 192.168.137.48, ::, fe80::72ee:50ff:f...",2880,2163,5043,3250963,3C:18:A0:41:C3:A0
1,HeimVision Smart WiFi Camera,44:01:BB:EC:10:4A,192.168.137.131,1430,761,2191,281445,3C:18:A0:41:C3:A0
17,Wyze Camera,7C:78:B2:86:0D:81,"0.0.0.0, 192.168.137.200",1177,782,1959,348353,3C:18:A0:41:C3:A0
57,TP-Link Tapo Camera,6C:5A:B0:44:1D:90,192.168.137.187,1014,811,1825,221768,3C:18:A0:41:C3:A0
26,SIMCAM 1S,10:2C:6B:1B:43:BE,192.168.137.179,854,790,1644,123823,3C:18:A0:41:C3:A0
39,Home Eye Camera,34:75:63:73:F3:36,192.168.137.173,901,658,1559,113405,3C:18:A0:41:C3:A0


# 6. 12개 파일전부 분석

In [19]:
all_device_results = []

for file in pcap_files:

    print("분석중:", file.name)

    total_packets, temp_df = analyze_pcap_devices(file)

    temp_df["file"] = file.name

    all_device_results.append(temp_df)


all_devices_df = pd.concat(
    all_device_results,
    ignore_index=True
)

분석중: DDoS-ACK_Fragmentation.pcap
분석중: DDoS-HTTP_Flood-.pcap
분석중: DDoS-ICMP_Flood.pcap
분석중: DDoS-ICMP_Fragmentation.pcap
분석중: DDoS-PSHACK_Flood.pcap
분석중: DDoS-RSTFINFlood.pcap
분석중: DDoS-SYN_Flood.pcap
분석중: DDoS-SlowLoris.pcap
분석중: DDoS-SynonymousIP_Flood.pcap
분석중: DDoS-TCP_Flood.pcap
분석중: DDoS-UDP_Flood.pcap
분석중: DDoS-UDP_Fragmentation.pcap


all_devices_df까지 만든 후 각 공격 파일에서 가장 많이 패킷 받은 top5확인하기.
12개 파일 모두 ddos공격이고 benign파일이 하나도 없다 ->공식CICIoT2023 설명에도
DDoS 유형으로 ACK Fragmentation, UDP Flood, SlowLoris, ICMP Flood, RSTFIN Flood, PSHACK Flood, HTTP Flood, UDP Fragmentation, TCP Flood, SYN Flood, SynonymousIP Flood 등이
포함되어 있는데, 내 폴더는 여기에 ICMP Fragmentation까지 포함된 ddos공격 묶음.

그래서 이 12개파일만 가지로 ddos인지 정상인지를 rf로 학습 못함. 왜냐? 전무 정답이 ddos=1 인셈.


## 1단계 — 지금 가진 DDoS PCAP 12개로 네트워크 구조 EDA
는 그대로 진행하기.

목표는
각 공격 PCAP   
 ↓   
어떤 MAC들이 등장했는가?   
 ↓   
각 MAC은 어떤 IP를 사용했는가?   
 ↓   
송신/수신 packet 수   
 ↓   
어떤 상대와 가장 많이 통신했는가?   
 ↓   
실제 IoT 기기 이름과 MAC 매칭   
 ↓   
공격별 가장 통신량 많은 기기 확인   

즉, analyze_pcap_devices() 코드부터 실행


단, 가장 많이 등장하는 mac= 가장 많이 공격받은 iot기기 라고 단정 x :  ddos에서는 공격자 기기들의 트래픽도 매우 많을 수 있음

그래서 피해기기만 찾기 위해 received_packets순위를 확인해야함.



In [20]:
top_received_per_file = (
    all_devices_df
    .sort_values(
        ["file", "received_packets"],
        ascending=[True, False]
    )
    .groupby("file")
    .head(5)
)

#각 공격 파일에서 가장 많이 패킷 받은 mac top 5나옴
top_received_per_file[
    [
        "file",
        "mac",
        "ips",
        "sent_packets",
        "received_packets",
        "total_packets"
    ]
]

,file,mac,ips,sent_packets,received_packets,total_packets
0,DDoS-ACK_Fragmentation.pcap,B0:C5:54:59:2E:99,192.168.137.132,8278,878305,886583
4,DDoS-ACK_Fragmentation.pcap,9C:8E:CD:1D:AB:9F,192.168.137.120,12678,564673,577351
1,DDoS-ACK_Fragmentation.pcap,8C:85:80:6C:B6:47,"192.168.137.123, fe80::8e85:80ff:fe6c:b647",268504,544412,812916
3,DDoS-ACK_Fragmentation.pcap,DC:A6:32:DC:27:D5,"0.0.0.0, 192.168.137.65, fd5a:7bd:2aea:e111:de...",486229,114608,600837
10,DDoS-ACK_Fragmentation.pcap,50:02:91:B1:68:0C,"192.168.137.223, ::",419,96547,96966
108,DDoS-HTTP_Flood-.pcap,9C:8E:CD:1D:AB:9F,"0.0.0.0, 192.168.137.139, 192.168.137.86",212128,475270,687398
109,DDoS-HTTP_Flood-.pcap,DC:A6:32:DC:27:D5,"192.168.137.178, 192.168.137.82, fe80::dea6:32...",197686,394220,591906
115,DDoS-HTTP_Flood-.pcap,DC:A6:32:C9:E5:A4,"192.168.137.186, 192.168.137.60, fe80::dea6:32...",40788,296704,337492
111,DDoS-HTTP_Flood-.pcap,DC:A6:32:C9:E4:90,"192.168.137.17, 192.168.137.238, fe80::dea6:32...",100548,278364,378912
112,DDoS-HTTP_Flood-.pcap,E4:5F:01:55:90:C4,"192.168.137.12, 192.168.137.89, fe80::e65f:1ff...",113287,261000,374287


In [21]:
# 반대로 송신량을 위에랑 비교해야함->비교로 많이 통신한 것 구별하기. !!!!!!

top_sent_per_file = (
    all_devices_df
    .sort_values(
        ["file", "sent_packets"],
        ascending=[True, False]
    )
    .groupby("file")
    .head(5)
)

top_sent_per_file[
    [
        "file",
        "mac",
        "ips",
        "sent_packets",
        "received_packets",
        "total_packets"
    ]
]

,file,mac,ips,sent_packets,received_packets,total_packets
2,DDoS-ACK_Fragmentation.pcap,DC:A6:32:C9:E4:AB,"0.0.0.0, 192.168.137.196, fd5a:7bd:2aea:e111:d...",555586,56391,611977
3,DDoS-ACK_Fragmentation.pcap,DC:A6:32:DC:27:D5,"0.0.0.0, 192.168.137.65, fd5a:7bd:2aea:e111:de...",486229,114608,600837
5,DDoS-ACK_Fragmentation.pcap,DC:A6:32:C9:E4:90,"0.0.0.0, 192.168.137.17, fe80::dea6:32ff:fec9:...",360659,31348,392007
6,DDoS-ACK_Fragmentation.pcap,DC:A6:32:C9:E5:EF,"0.0.0.0, 192.168.137.42, fd5a:7bd:2aea:e111:de...",268720,27981,296701
1,DDoS-ACK_Fragmentation.pcap,8C:85:80:6C:B6:47,"192.168.137.123, fe80::8e85:80ff:fe6c:b647",268504,544412,812916
114,DDoS-HTTP_Flood-.pcap,B0:F1:EC:D3:E7:98,"0.0.0.0, 192.168.137.150, fe80::b2f1:ecff:fed3...",341724,0,341724
117,DDoS-HTTP_Flood-.pcap,DC:A6:32:C9:E5:02,"0.0.0.0, 192.168.137.134, 192.168.137.44, fd5a...",291356,9,291365
119,DDoS-HTTP_Flood-.pcap,DC:A6:32:C9:E4:C6,"0.0.0.0, 192.168.137.133, 192.168.137.156, ::,...",280682,3,280685
120,DDoS-HTTP_Flood-.pcap,DC:A6:32:C9:E6:F4,"0.0.0.0, 192.168.137.114, 192.168.137.147, fd5...",271981,4,271985
108,DDoS-HTTP_Flood-.pcap,9C:8E:CD:1D:AB:9F,"0.0.0.0, 192.168.137.139, 192.168.137.86",212128,475270,687398


## 2단계: 카메라가 실제 피해 기기인지 확인하기

## 3단계 rf용 대이터는 따로 준비하기:

### 3방법 a공식 CICIoT2023csv파일 사용 -> 이미 pcap에서 ml특징 추출한 csv가 있고 공식 페이지에서도 이 csv를 ml평가용으로 함

우리 PCAP
→ 기기별 네트워크 분석용

공식 CSV
→ RF 학습용

으로 역할 나누기.

###3방법b 우리가 PCAP에서 직접 feature extraction

직접 flow_duration,
Rate,
Srate,
Drate,
IAT,
syn_count,
ack_count,
packet length statistics,
...
를 계산해야하고 공식 피쳐 extraction과 동일한 윈도우 정의까지 재현해야하는데 비효율적

In [22]:

# RF 넘어가기전 앞서 만든 analyze_pcap_devices()를 12개에 돌린 뒤, 아래 세 결과 뽑기

# 1. 파일별 total_packets TOP 5

top_total = (
    all_devices_df
    .sort_values(
        ["file", "total_packets"],
        ascending=[True, False]
    )
    .groupby("file")
    .head(5)
)


# 2. 파일별 received_packets TOP 5

top_received = (
    all_devices_df
    .sort_values(
        ["file", "received_packets"],
        ascending=[True, False]
    )
    .groupby("file")
    .head(5)
)


# 3. 파일별 sent_packets TOP 5

top_sent = (
    all_devices_df
    .sort_values(
        ["file", "sent_packets"],
        ascending=[True, False]
    )
    .groupby("file")
    .head(5)
)

In [23]:
file_name = "DDoS-SYN_Flood.pcap"

all_devices_df[
    all_devices_df["file"] == file_name
][
    [
        "mac",
        "ips",
        "sent_packets",
        "received_packets",
        "total_packets",
        "top_peer"
    ]
].sort_values(
    "received_packets",
    ascending=False
).head(20)

,mac,ips,sent_packets,received_packets,total_packets,top_peer
632,1C:12:B0:9B:0C:EC,"0.0.0.0, 192.168.137.99, ::, fe80::1e12:b0ff:f...",170629,8786693,8957322,DC:A6:32:C9:E4:AB
633,1C:FE:2B:98:16:DD,"0.0.0.0, 192.168.137.30",223059,7655917,7878976,DC:A6:32:C9:E4:AB
635,CC:F4:11:9C:D0:00,"0.0.0.0, 192.168.137.12, ::, fe80::cef4:11ff:f...",5618,6701211,6706829,DC:A6:32:C9:E4:AB
641,9C:8E:CD:1D:AB:9F,192.168.137.161,14686,2870769,2885455,DC:A6:32:C9:E4:AB
634,DC:A6:32:C9:E4:AB,"192.168.137.240, fe80::dea6:32ff:fec9:e4ab",7079462,87516,7166978,1C:12:B0:9B:0C:EC
642,3C:18:A0:41:C3:A0,"1.1.1.1, 100.20.126.204, 100.20.212.253, 100.2...",81589,75973,157562,AC:17:02:05:34:27
636,DC:A6:32:DC:27:D5,"192.168.137.17, fe80::dea6:32ff:fedc:27d5",4307275,70868,4378143,1C:12:B0:9B:0C:EC
637,DC:A6:32:C9:E4:90,192.168.137.195,4163427,64320,4227747,1C:FE:2B:98:16:DD
638,E4:5F:01:55:90:C4,192.168.137.134,3827597,60845,3888442,1C:FE:2B:98:16:DD
639,DC:A6:32:C9:E5:EF,"192.168.137.58, fe80::dea6:32ff:fec9:e5ef",3659456,56798,3716254,1C:FE:2B:98:16:DD


# 결과저장 폴더 만들기
이미 결과 CSV가 있는 PCAP은 자동으로 건너뜀

In [24]:
from pathlib import Path

RESULT_DIR = PCAP_DIR / "analysis_results"
RESULT_DIR.mkdir(exist_ok=True)

print(RESULT_DIR)

/content/drive/MyDrive/CICIoT2023/analysis_results


In [25]:
import pandas as pd

for i, file in enumerate(pcap_files, start=1):

    output_file = RESULT_DIR / f"{file.stem}_device_summary.csv"

    print(f"\n[{i}/{len(pcap_files)}] {file.name}")

    # 이미 분석 결과가 있으면 건너뛰기
    if output_file.exists():
        print("→ 이미 분석 완료. 건너뜀")
        continue

    print("→ 분석 시작")

    total_packets, temp_df = analyze_pcap_devices(file)

    temp_df["file"] = file.name
    temp_df["pcap_total_packets"] = total_packets

    # 분석 직후 즉시 저장
    temp_df.to_csv(
        output_file,
        index=False
    )

    print("→ 분석 완료")
    print("패킷 수:", f"{total_packets:,}")
    print("저장:", output_file)


[1/12] DDoS-ACK_Fragmentation.pcap
→ 이미 분석 완료. 건너뜀

[2/12] DDoS-HTTP_Flood-.pcap
→ 이미 분석 완료. 건너뜀

[3/12] DDoS-ICMP_Flood.pcap
→ 이미 분석 완료. 건너뜀

[4/12] DDoS-ICMP_Fragmentation.pcap
→ 이미 분석 완료. 건너뜀

[5/12] DDoS-PSHACK_Flood.pcap
→ 이미 분석 완료. 건너뜀

[6/12] DDoS-RSTFINFlood.pcap
→ 이미 분석 완료. 건너뜀

[7/12] DDoS-SYN_Flood.pcap
→ 이미 분석 완료. 건너뜀

[8/12] DDoS-SlowLoris.pcap
→ 이미 분석 완료. 건너뜀

[9/12] DDoS-SynonymousIP_Flood.pcap
→ 이미 분석 완료. 건너뜀

[10/12] DDoS-TCP_Flood.pcap
→ 이미 분석 완료. 건너뜀

[11/12] DDoS-UDP_Flood.pcap
→ 이미 분석 완료. 건너뜀

[12/12] DDoS-UDP_Fragmentation.pcap
→ 이미 분석 완료. 건너뜀


In [30]:
all_results = []

for file in result_files:
    df_temp = pd.read_csv(file)
    all_results.append(df_temp)

all_devices_df = pd.concat(
    all_results,
    ignore_index=True
)

print(all_devices_df.shape)

all_devices_df.head()

NameError: name 'result_files' is not defined

In [28]:
all_devices_df.to_csv(
    RESULT_DIR / "all_devices_summary.csv",
    index=False
)


## 다시 분석 안하게 저장해둬서 다음번 코랩에서는 다음 코드만 실행하면됨


`all_devices_df.to_csv(
    RESULT_DIR / "all_devices_summary.csv",
    index=False
)`